# Starbucks Promotion Optimization
## Phase 3 — Uplift Modeling (Causal ML)

**Goal:** estimate the **Conditional Average Treatment Effect (CATE)** of an offer for each customer — *how much extra spend does seeing this offer cause, for this kind of customer?* No propensity-to-buy classifier is built to target customers.

### Design decisions (each is justified in the cell where it is implemented)

| Question | Decision |
|---|---|
| **Which offers?** | BOGO and discount offers only (61,042 instances). Informational offers have no reward, so there is no discount cost to optimise in Phase 4. |
| **Target** | **Incremental revenue** — customer spend in the outcome window — rather than incremental conversion probability. |
| **Treatment** | *Viewed the offer within 24 hours of receiving it.* Control = had not viewed it by then. |
| **Outcome window** | Hours **24 to 72** after receipt. Starts at the treatment decision point and ends before the next offer wave. |
| **Learners** | **T-Learner** (two models) and **S-Learner** (one model), both gradient-boosted trees. |
| **Validation** | **Time-based only.** Train on the first 20 days (hour < 480), validate on the last 10 days. Hyper-parameters chosen by forward-chaining CV *inside* the training period. |

### Why the treatment is defined with a "landmark" (a refinement of the Phase 1 rule)
Phase 1 put accidental completers into control by looking at **when the customer completed** the offer. That is fine for describing the funnel, but for modelling it has a flaw: *completing early means having spent a lot*, so the control group would be filled with high spenders **because of the outcome we are trying to measure**. A landmark design avoids this: treatment is decided from what happened **before hour 24**, and the outcome is measured **only after hour 24**. The principle from Phase 1 is kept (an offer a customer had not seen cannot cause a purchase), but the group definition no longer depends on the spend we are predicting. A comparison against the Phase 1 rule is shown below.

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("../outputs")
FIG_DIR = Path("../figures/phase3"); FIG_DIR.mkdir(parents=True, exist_ok=True)

LANDMARK_H     = 24     # treatment = viewed before this many hours after receipt
OUTCOME_END_H  = 72     # outcome window = [LANDMARK_H, OUTCOME_END_H) hours after receipt
TRAIN_CUTOFF_H = 480    # first 20 days (hour < 480) = train ; last 10 days = validation
WINSOR_Q       = 0.99   # cap extreme spend at the 99th percentile of the TRAINING data
N_BOOT         = 300
SEED           = 42
DATA_END_H     = 714    # last event in the transcript

PARAM_GRID = {
    "shallow":   dict(max_depth=2, learning_rate=0.05, max_iter=150, min_samples_leaf=200, l2_regularization=5.0),
    "medium":    dict(max_depth=3, learning_rate=0.05, max_iter=150, min_samples_leaf=200, l2_regularization=5.0),
    "longer":    dict(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=100, l2_regularization=5.0),
    "deep_slow": dict(max_depth=4, learning_rate=0.03, max_iter=200, min_samples_leaf=300, l2_regularization=5.0),
}
print("Config loaded.")

In [ ]:
# ============================================================
# LOAD PHASE 1 + PHASE 2 OUTPUTS
# ============================================================
model_table = pd.read_csv(OUTPUT_DIR / "phase2_model_table.csv")
tx   = pd.read_csv(OUTPUT_DIR / "attributed_transactions.csv")[["person_id", "transaction_time", "amount"]]
prof = pd.read_csv(OUTPUT_DIR / "clean_profile.csv")[["person_id", "age", "income", "gender", "demo_missing"]]

df = (model_table[model_table["offer_type"].isin(["bogo", "discount"])]
      .merge(prof, on="person_id", how="left", validate="many_to_one")
      .reset_index(drop=True))

# each customer receives at most one offer per wave -> outcome windows of one customer never overlap
assert df.groupby(["person_id", "time_received"]).size().max() == 1
print("All offer instances      :", f"{len(model_table):,}")
print("BOGO + discount instances:", f"{len(df):,}")
print("Waves (hours)            :", sorted(df['time_received'].unique()))

## 1. Target definition — incremental revenue, not incremental conversion

**Choice:** the dependent variable is the customer's **spend in the outcome window** (dollars). The model estimates `E[spend | offer seen, x] − E[spend | offer not seen, x]`.

**Why revenue and not conversion probability**
1. **It is the business objective.** Phase 4 maximises *Net Incremental Revenue* under a dollar budget. A conversion model treats a $2 purchase and a $40 purchase as the same "success".
2. **Discounts change basket size as well as purchase frequency.** A conversion target cannot see someone buying more per visit.
3. **"Conversion" is ambiguous here.** BOGO and discount offers have different completion thresholds ($5-$20), so a conversion label would not mean the same thing across offers.

**Costs of the choice:** spend is right-skewed with many zeros (about half the customers spend nothing in a 48-hour window). To stop a few very large baskets dominating the fit, spend is **capped at the training-period 99th percentile** (cap computed from training rows only). Results on uncapped spend are reported as a robustness check.

**Outcome window — hours 24 to 72 after receipt**
- It starts at the landmark, so nothing that happened before the treatment decision is counted as an effect of treatment.
- It ends 72 hours after receipt. Offer waves are at least 72 hours apart, so a window never overlaps the next wave. That is what lets the time-based split below be *completely* free of overlap.

In [ ]:
# ============================================================
# OUTCOME: spend in [24h, 72h) after the offer is received
# ============================================================
def spend_between(df, tx, start_h, end_h):
    """Total spend with  time_received + start_h <= transaction_time < time_received + end_h."""
    tx_by = {p: (g["transaction_time"].values, np.concatenate([[0.0], g["amount"].values.cumsum()]))
             for p, g in tx.sort_values(["person_id", "transaction_time"]).groupby("person_id")}
    out = np.zeros(len(df))
    for i, (p, s) in enumerate(zip(df["person_id"], df["time_received"])):
        if p in tx_by:
            t, cs = tx_by[p]
            out[i] = cs[np.searchsorted(t, s + end_h, "left")] - cs[np.searchsorted(t, s + start_h, "left")]
    return out

df["y_raw"] = spend_between(df, tx, LANDMARK_H, OUTCOME_END_H)

# every outcome window must lie fully inside the observed data (no right-censoring)
assert (df["time_received"] + OUTCOME_END_H <= DATA_END_H).all()

print(df["y_raw"].describe().round(2).to_string())
print(f"\nShare with zero spend in the window: {(df['y_raw'] == 0).mean():.1%}")

## 2. Treatment definition — the landmark rule

`treated = 1` if the customer **viewed the offer before hour 24** after receiving it; `treated = 0` otherwise (viewed later, or never).

- Accidental completers who never viewed the offer are **control** — consistent with Phase 1.
- Customers who view the offer *after* the landmark are also control. This slightly dilutes the contrast (some controls eventually see the offer), so effects are, if anything, **understated**.
- Group membership is fixed from information available **before** any outcome is measured.

**Important caveat:** viewing is a customer's own choice, not randomised. The model therefore estimates a *conditional association between seeing the offer and spending*, adjusted for the features. Section 7 checks how strongly viewers and non-viewers differ and uses inverse-propensity weights as a robustness check.

In [ ]:
# ============================================================
# TREATMENT FLAG
# ============================================================
df["treated"] = (df["time_viewed"] < df["time_received"] + LANDMARK_H).astype(int)   # NaN (never viewed) -> False

print(f"Treated share overall: {df['treated'].mean():.1%}")
print(f"Never viewed at all  : {df['time_viewed'].isna().mean():.1%}")
print(f"Viewed, but after the landmark: {((df['treated'] == 0) & df['time_viewed'].notna()).mean():.1%}")

In [ ]:
# ============================================================
# WHY NOT THE PHASE 1 RULE? Compare how stable the naive effect is across waves.
# Phase 1 rule (A): treated = viewed within the window and NOT viewed only after completing;
# outcome = spend over the whole window [0h, 72h).
# ============================================================
df["y_full"] = spend_between(df, tx, 0, OUTCOME_END_H)
in_window = df["time_viewed"].notna() & (df["time_viewed"] < df["time_received"] + OUTCOME_END_H)
viewed_after_done = (df["completed"] == 1) & (df["time_viewed"] > df["time_completed"].fillna(np.inf))
df["treated_rule_A"] = (in_window & ~viewed_after_done).astype(int)

def wave_gap(d, y, w):
    return d.groupby("time_received").apply(lambda g: g.loc[g[w] == 1, y].mean() - g.loc[g[w] == 0, y].mean())

cmp = pd.DataFrame({
    "landmark design: treated-minus-control $": wave_gap(df, "y_raw", "treated"),
    "Phase 1 rule A, whole window: treated-minus-control $": wave_gap(df, "y_full", "treated_rule_A"),
    "share treated (landmark)": df.groupby("time_received")["treated"].mean(),
    "share treated (rule A)": df.groupby("time_received")["treated_rule_A"].mean(),
}).round(2)
display(cmp)
print("The landmark design gives a steady effect and a steady treated share from wave to wave.")
print("Rule A drifts wave to wave: its control group fills with early completers, i.e. it is defined partly by the outcome.")

## 3. Features (all measured at the moment the offer arrives)

Phase 2 features plus offer design and demographics. **Cumulative counts and spends are converted to per-day rates** (divided by days elapsed). Without that, later waves would always look "bigger" than anything seen in training, which is a time-shift problem, not behaviour.

Columns that describe what happened *after* receipt are explicitly **banned** and checked by an assertion: `time_viewed`, `time_completed`, `viewed`, `completed`, `view_status`, `accidental_completion`, `window_spend`, and every outcome column.

In [ ]:
# ============================================================
# FEATURE MATRIX
# ============================================================
days = (df["time_received"] / 24).replace(0, np.nan)          # wave 0: no history -> rates are empty

df["is_bogo"]             = (df["offer_type"] == "bogo").astype(int)
df["tx_rate_per_day"]     = df["n_prior_tx"] / days
df["spend_rate_per_day"]  = (df["prior_offer_period_spend"] + df["prior_organic_spend"]) / days
df["offer_exposure_share"] = df["prior_offer_exposure_days"] / days
df["gender"]              = df["gender"].fillna("U")
for g in ["F", "M", "O", "U"]:
    df[f"gender_{g}"] = (df["gender"] == g).astype(int)
df["demo_missing"]        = df["demo_missing"].astype(int)

FAMILIES = {
    "Offer design":        ["is_bogo", "difficulty", "duration", "offer_reward"],
    "Offer fatigue":       ["offers_received_14d", "offers_received_30d"],
    "Inter-purchase time": ["ipt_days", "ipt_days_last3", "days_since_last_organic_tx"],
    "Reward hunting":      ["reward_hunting_index", "reward_hunting_index_per_day", "offer_exposure_share"],
    "Time-decayed AOV":    ["aov_decayed_hl7d"],
    "Purchase history":    ["has_purchase_history", "tx_rate_per_day", "spend_rate_per_day"],
    "Demographics":        ["age", "income", "gender_F", "gender_M", "gender_O", "gender_U", "demo_missing"],
}
FEATURES = [c for cols in FAMILIES.values() for c in cols]

BANNED = {"time_viewed", "time_completed", "viewed", "completed", "view_status", "accidental_completion",
          "window_spend", "y_raw", "y_full", "y", "treated", "treated_rule_A", "time_received", "instance_id"}
assert not (set(FEATURES) & BANNED), set(FEATURES) & BANNED
assert len(FEATURES) == len(set(FEATURES))
print(f"{len(FEATURES)} features:")
for k, v in FAMILIES.items():
    print(f"  {k:20s}: {v}")

## 4. Leakage-proof, time-based validation

Random `train_test_split` is **not used anywhere**. Offers go out in waves at hours 0, 168, 336, 408, 504 and 576.

- **Final split:** train on waves 0, 168, 336, 408 (first 20 days, hour < 480); validate on waves 504 and 576 (last 10 days).
- **Hyper-parameter selection:** forward-chaining CV *inside the training period only* — fold 1 trains on waves 0-168 and validates on wave 336; fold 2 trains on waves 0-336 and validates on wave 408. The final validation waves are never touched during tuning.
- **No label overlap:** an outcome window ends 72 hours after its wave. Every training window must therefore end *before* the first validation wave starts. This is asserted below.
- The winsorisation cap is recomputed from the training rows of each fold.

In [ ]:
# ============================================================
# TIME-BASED SPLITS + NO-OVERLAP ASSERTIONS
# ============================================================
cv_folds = [
    {"name": "fold 1", "train_waves": [0, 168],      "val_waves": [336]},
    {"name": "fold 2", "train_waves": [0, 168, 336], "val_waves": [408]},
]
train_df   = df[df["time_received"] <  TRAIN_CUTOFF_H].reset_index(drop=True)
holdout_df = df[df["time_received"] >= TRAIN_CUTOFF_H].reset_index(drop=True)

rows = []
for f in cv_folds:
    last_label_end = max(f["train_waves"]) + OUTCOME_END_H
    first_val      = min(f["val_waves"])
    assert last_label_end <= first_val, f"label overlap in {f['name']}"
    rows.append((f["name"], f["train_waves"], f["val_waves"], last_label_end, first_val))

last_label_end = train_df["time_received"].max() + OUTCOME_END_H
assert last_label_end <= holdout_df["time_received"].min() and last_label_end <= TRAIN_CUTOFF_H
rows.append(("FINAL", sorted(train_df["time_received"].unique()), sorted(holdout_df["time_received"].unique()),
             last_label_end, holdout_df["time_received"].min()))

display(pd.DataFrame(rows, columns=["split", "train waves", "validation waves",
                                    "last training label ends (h)", "first validation offer (h)"]))
print(f"Training rows: {len(train_df):,}   Final validation rows: {len(holdout_df):,}")
print("All assertions passed: no training outcome is measured after a validation offer is sent.")

## 5. Algorithms

**T-Learner (two-model):** model A is trained **only on control** customers, model B **only on treated** customers. Uplift = `B(x) − A(x)`.

**S-Learner (single-model):** one model is trained on everyone with `treated` as an input feature. To score a customer it is run twice (`treated = 1` and `treated = 0`) and the predictions are subtracted.

Both use gradient-boosted regression trees (`HistGradientBoostingRegressor`, which handles the empty values in IPT and per-day features natively). The target is capped spend in dollars, so the output is **incremental dollars per customer**.

In [ ]:
# ============================================================
# LEARNERS
# ============================================================
def make_model(params):
    return HistGradientBoostingRegressor(random_state=SEED, **params)

class TLearner:
    name = "T-Learner"
    def __init__(self, params): self.params = params
    def fit(self, X, w, y):
        self.m0 = make_model(self.params).fit(X[w == 0], y[w == 0])      # control only
        self.m1 = make_model(self.params).fit(X[w == 1], y[w == 1])      # treated only
        return self
    def predict(self, X):
        return self.m1.predict(X) - self.m0.predict(X)

class SLearner:
    name = "S-Learner"
    def __init__(self, params): self.params = params
    def fit(self, X, w, y):
        self.m = make_model(self.params).fit(X.assign(_treated=w), y)
        return self
    def predict(self, X):
        return self.m.predict(X.assign(_treated=1)) - self.m.predict(X.assign(_treated=0))

def cap_from(train_rows):
    """Winsorisation cap computed from TRAINING rows only."""
    return train_rows["y_raw"].quantile(WINSOR_Q)

## 6. Evaluation tools for uplift (no accuracy / ROC-AUC)

Customers are ranked by predicted uplift. Among the top *k* the **incremental revenue curve** is
`k × (mean spend of treated − mean spend of control)` — the dollars gained by targeting the top *k*. A model with no ranking skill follows the straight *random-targeting* line. The **Qini coefficient** used here is the average gap between the model curve and the random line, per customer (dollars; positive = better than random).

Because viewing isn't randomised, an **inverse-propensity-weighted (IPW)** version is also reported (Section 7).

In [ ]:
# ============================================================
# UPLIFT EVALUATION FUNCTIONS
# ============================================================
def _grid(n, n_points=100):
    return np.unique(np.ceil(np.linspace(0.01, 1.0, n_points) * n).astype(int))

def incremental_curve(u, w, y, n_points=100):
    """k * (mean treated - mean control) among the top-k by predicted uplift."""
    o = np.argsort(-u, kind="stable"); w, y = w[o], y[o]
    n = len(y); ks = _grid(n, n_points); k = ks - 1
    c1, c0 = np.cumsum(w), np.cumsum(1 - w)
    s1, s0 = np.cumsum(y * w), np.cumsum(y * (1 - w))
    with np.errstate(divide="ignore", invalid="ignore"):
        inc = ks * (s1[k] / c1[k] - s0[k] / c0[k])
    return ks / n, inc

def qini_coef(u, w, y):
    frac, inc = incremental_curve(u, w, y)
    return np.nanmean(inc - frac * inc[-1]) / len(y)

def ipw_curve(u, w, y, e, n_points=100):
    o = np.argsort(-u, kind="stable")
    n = len(y); ks = _grid(n, n_points)
    z = ((w / e - (1 - w) / (1 - e)) * y)[o]
    return ks / n, np.cumsum(z)[ks - 1]

def ipw_qini_coef(u, w, y, e):
    frac, g = ipw_curve(u, w, y, e)
    return np.mean(g - frac * g[-1]) / len(y)

def bootstrap_ci(fn, n, B=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = [fn(rng.integers(0, n, n)) for _ in range(B)]
    return np.percentile(vals, [2.5, 97.5])

def decile_table(u, w, y, e=None):
    d = pd.DataFrame({"u": u, "w": w, "y": y})
    d["decile"] = 10 - pd.qcut(d["u"].rank(method="first", ascending=False), 10, labels=False)   # 10 = highest predicted uplift
    d["decile"] = pd.qcut(d["u"].rank(method="first"), 10, labels=False) + 1                     # 1 = lowest ... 10 = highest
    rows = []
    for k, g in d.groupby("decile"):
        t, c = g[g.w == 1]["y"], g[g.w == 0]["y"]
        diff = t.mean() - c.mean()
        se = np.sqrt(t.var() / len(t) + c.var() / len(c))
        row = {"decile": k, "n": len(g), "mean predicted uplift $": g["u"].mean(),
               "observed uplift $": diff, "ci_low": diff - 1.96 * se, "ci_high": diff + 1.96 * se}
        if e is not None:
            ge = e[g.index]; wt = g["w"].values
            row["IPW observed uplift $"] = (np.sum(wt * g["y"] / ge) / np.sum(wt / ge)
                                           - np.sum((1 - wt) * g["y"] / (1 - ge)) / np.sum((1 - wt) / (1 - ge)))
        rows.append(row)
    return pd.DataFrame(rows).round(2)
print("Evaluation helpers ready.")

## 7. Time-based cross-validation — choosing hyper-parameters

Each configuration is trained on the earlier waves and scored on the next wave, for both learners. The Qini coefficient (average across the two folds) picks the configuration. **The final validation waves are not used here.**

In [ ]:
# ============================================================
# FORWARD-CHAINING CV (inside the training period only)
# ============================================================
cv_rows = []
for f in cv_folds:
    tr = train_df[train_df["time_received"].isin(f["train_waves"])]
    va = train_df[train_df["time_received"].isin(f["val_waves"])]
    cap = cap_from(tr)                                           # cap from fold-training rows only
    ytr, yva = tr["y_raw"].clip(upper=cap).values, va["y_raw"].clip(upper=cap).values
    for cfg, params in PARAM_GRID.items():
        for L in (TLearner, SLearner):
            model = L(params).fit(tr[FEATURES], tr["treated"].values, ytr)
            u = model.predict(va[FEATURES])
            cv_rows.append({"fold": f["name"], "learner": L.name, "config": cfg,
                            "qini_coef": qini_coef(u, va["treated"].values, yva)})

cv = pd.DataFrame(cv_rows)
cv_summary = (cv.pivot_table(index=["learner", "config"], columns="fold", values="qini_coef")
                .assign(mean=lambda d: d.mean(axis=1)).round(3))
display(cv_summary)

best_cfg = {L: cv_summary.loc[L]["mean"].idxmax() for L in ["T-Learner", "S-Learner"]}
print("Selected configuration per learner (highest mean CV Qini):", best_cfg)

## 8. Final models — trained on the first 20 days, scored on the last 10

Everything in this section uses the final time split. The propensity model below predicts **whether a customer views the offer within 24 hours** (not whether they buy) and is used **only to re-weight the evaluation**; it never selects customers.

In [ ]:
# ============================================================
# FINAL FIT ON THE TRAINING PERIOD, SCORE THE HOLD-OUT (last 10 days)
# ============================================================
CAP = cap_from(train_df)
train_df["y"]   = train_df["y_raw"].clip(upper=CAP)
holdout_df["y"] = holdout_df["y_raw"].clip(upper=CAP)
print(f"Spend cap (99th percentile of training period): ${CAP:.2f}")

Xtr, wtr, ytr = train_df[FEATURES], train_df["treated"].values, train_df["y"].values
Xho, who, yho = holdout_df[FEATURES], holdout_df["treated"].values, holdout_df["y"].values

models = {
    "T-Learner": TLearner(PARAM_GRID[best_cfg["T-Learner"]]).fit(Xtr, wtr, ytr),
    "S-Learner": SLearner(PARAM_GRID[best_cfg["S-Learner"]]).fit(Xtr, wtr, ytr),
}
holdout_df["uplift_t"] = models["T-Learner"].predict(Xho)
holdout_df["uplift_s"] = models["S-Learner"].predict(Xho)

# ---- overlap diagnostic + evaluation weights: propensity to VIEW within 24h (not to buy) ----
prop_model = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                            min_samples_leaf=200, random_state=SEED).fit(Xtr, wtr)
e_raw = prop_model.predict_proba(Xho)[:, 1]
e = np.clip(e_raw, 0.05, 0.95)
holdout_df["propensity_view"] = e
print(f"Propensity-to-view AUC on hold-out: {roc_auc_score(who, e_raw):.3f}")
print(f"Share of hold-out with propensity inside [0.05, 0.95] (overlap): {((e_raw > 0.05) & (e_raw < 0.95)).mean():.1%}")

In [ ]:
# ============================================================
# HOLD-OUT RESULTS: Qini coefficient (+95% bootstrap CI), naive and IPW-weighted
# ============================================================
rows = []
raw_y = holdout_df["y_raw"].values
for name, col in [("T-Learner", "uplift_t"), ("S-Learner", "uplift_s")]:
    u = holdout_df[col].values
    q = qini_coef(u, who, yho);  q_ci = bootstrap_ci(lambda i: qini_coef(u[i], who[i], yho[i]), len(u))
    qi = ipw_qini_coef(u, who, yho, e);  qi_ci = bootstrap_ci(lambda i: ipw_qini_coef(u[i], who[i], yho[i], e[i]), len(u))
    qr = qini_coef(u, who, raw_y)
    rows.append({"model": name, "Qini coef $/customer": q, "95% CI": f"[{q_ci[0]:.3f}, {q_ci[1]:.3f}]",
                 "IPW Qini coef": qi, "IPW 95% CI": f"[{qi_ci[0]:.3f}, {qi_ci[1]:.3f}]",
                 "Qini on uncapped spend": qr,
                 "mean predicted uplift $": u.mean(),
                 "Spearman(T,S)": stats.spearmanr(holdout_df["uplift_t"], holdout_df["uplift_s"])[0]})
results = pd.DataFrame(rows).round(3)
display(results)

obs_ate = yho[who == 1].mean() - yho[who == 0].mean()
print(f"Observed hold-out treated-minus-control spend: ${obs_ate:.2f}   (random targeting = Qini coefficient 0)")

In [ ]:
# ============================================================
# DECILE CHECK: do customers ranked higher really show a larger observed effect?
# ============================================================
decile_results = {}
for name, col in [("T-Learner", "uplift_t"), ("S-Learner", "uplift_s")]:
    t = decile_table(holdout_df[col], pd.Series(who), pd.Series(yho), e)
    decile_results[name] = t
    rho, p = stats.spearmanr(t["decile"], t["observed uplift $"])
    top, bot = t.iloc[-1], t.iloc[0]
    print(f"\n===== {name} =====   Spearman(decile, observed uplift) = {rho:.2f} (p = {p:.3f})")
    display(t)
    print(f"Top decile observed uplift ${top['observed uplift $']:.2f}  vs bottom decile ${bot['observed uplift $']:.2f}")

In [ ]:
# ============================================================
# PLOT: predicted vs observed uplift by decile
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, t) in zip(axes, decile_results.items()):
    ax.bar(t["decile"], t["observed uplift $"], color="steelblue", alpha=0.8, label="observed (treated - control)")
    ax.errorbar(t["decile"], t["observed uplift $"],
                yerr=[t["observed uplift $"] - t["ci_low"], t["ci_high"] - t["observed uplift $"]], fmt="none", ecolor="black", capsize=3)
    ax.plot(t["decile"], t["mean predicted uplift $"], "o-", color="darkorange", label="mean predicted")
    ax.axhline(0, color="gray", lw=1)
    ax.set_title(f"{name} — last 10 days (hold-out)")
    ax.set_xlabel("Predicted-uplift decile (10 = highest)"); ax.set_ylabel("Uplift in spend ($)")
    ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "uplift_by_decile.png", dpi=120); plt.show()

## 9. Are there customers the offer does not help? ("Sleeping dogs" check)
Customers with a **negative predicted uplift** are the ones a model would flag as harmed by the offer. The test is whether their *observed* effect is genuinely negative.

In [ ]:
# ============================================================
# NEGATIVE-PREDICTED-UPLIFT GROUP
# ============================================================
rows = []
for name, col in [("T-Learner", "uplift_t"), ("S-Learner", "uplift_s")]:
    m = holdout_df[col] < 0
    if m.sum() < 50:
        rows.append({"model": name, "n flagged": int(m.sum()), "note": "too few to assess"}); continue
    t, c = yho[m & (who == 1)], yho[m & (who == 0)]
    diff = t.mean() - c.mean(); se = np.sqrt(t.var() / len(t) + c.var() / len(c))
    rows.append({"model": name, "n flagged": int(m.sum()), "share of hold-out": m.mean(),
                 "observed uplift $": diff, "95% CI": f"[{diff - 1.96*se:.2f}, {diff + 1.96*se:.2f}]",
                 "evidence of harm": (diff + 1.96 * se) < 0})
display(pd.DataFrame(rows).round(3))
print("'evidence of harm' is True only if the whole confidence interval is below zero.")

## 10. Which engineered features drive the uplift? (permutation importance)
Each feature (or feature family) is shuffled across customers in the hold-out, the uplift scores are recomputed, and the **drop in Qini coefficient** is recorded. A big drop means the ranking relied on that information. Repeated 5 times.

In [ ]:
# ============================================================
# PERMUTATION IMPORTANCE ON THE HOLD-OUT (drop in Qini coefficient)
# ============================================================
rng = np.random.default_rng(SEED)
imp_rows_family, imp_rows_feat = [], []
for name, model in models.items():
    base = qini_coef(model.predict(Xho), who, yho)
    def drop_for(cols, R=5):
        ds = []
        for _ in range(R):
            Xp = Xho.copy()
            perm = rng.permutation(len(Xp))
            Xp[cols] = Xho[cols].values[perm]
            ds.append(base - qini_coef(model.predict(Xp), who, yho))
        return np.mean(ds), np.std(ds)
    for fam, cols in FAMILIES.items():
        m, s = drop_for(cols); imp_rows_family.append({"model": name, "family": fam, "qini drop": m, "std": s})
    for c in FEATURES:
        m, s = drop_for([c], R=3); imp_rows_feat.append({"model": name, "feature": c, "qini drop": m})

fam_imp = (pd.DataFrame(imp_rows_family).pivot(index="family", columns="model", values="qini drop")
             .assign(average=lambda d: d.mean(axis=1)).sort_values("average", ascending=False).round(3))
feat_imp = (pd.DataFrame(imp_rows_feat).pivot(index="feature", columns="model", values="qini drop")
              .assign(average=lambda d: d.mean(axis=1)).sort_values("average", ascending=False).round(3))
print("By feature family:");  display(fam_imp)
print("Top 10 individual features:"); display(feat_imp.head(10))

In [ ]:
# ============================================================
# PLOT: family importance
# ============================================================
fam_imp["average"].sort_values().plot.barh(figsize=(8, 4), color="steelblue")
plt.xlabel("Average drop in Qini coefficient when shuffled"); plt.title("Which feature families drive uplift ranking?")
plt.tight_layout(); plt.savefig(FIG_DIR / "family_importance.png", dpi=120); plt.show()

## 11. Leakage and sanity audit
1. **Banned columns** were asserted absent from the features (Section 3).
2. **No label overlap** between training and validation was asserted (Section 4).
3. **Placebo test:** if the treatment labels in the *training* data are shuffled within each wave, the models can learn nothing real about who responds. Their Qini coefficient on the true hold-out should then be about zero. A model that still scored well would be leaking.

In [ ]:
# ============================================================
# PLACEBO TEST — shuffle training treatment labels within each wave
# ============================================================
rng = np.random.default_rng(SEED)
w_placebo = train_df.groupby("time_received")["treated"].transform(lambda s: rng.permutation(s.values)).values

rows = []
for L, key in [(TLearner, "T-Learner"), (SLearner, "S-Learner")]:
    pm = L(PARAM_GRID[best_cfg[key]]).fit(Xtr, w_placebo, ytr)
    u = pm.predict(Xho)
    q = qini_coef(u, who, yho); ci = bootstrap_ci(lambda i: qini_coef(u[i], who[i], yho[i]), len(u))
    real = results.set_index("model").loc[key, "Qini coef $/customer"]
    rows.append({"model": key, "placebo Qini": q, "placebo 95% CI": f"[{ci[0]:.3f}, {ci[1]:.3f}]",
                 "real Qini": real, "placebo CI contains 0": bool(ci[0] <= 0 <= ci[1])})
display(pd.DataFrame(rows).round(3))

## 12. Save outputs for Phase 4

In [ ]:
# ============================================================
# SAVE
# ============================================================
keep = ["instance_id", "person_id", "offer_id", "offer_type", "time_received", "offer_reward", "difficulty", "duration",
        "treated", "y_raw", "y", "completed", "uplift_t", "uplift_s", "propensity_view"]
holdout_df[keep].to_csv(OUTPUT_DIR / "phase3_scored_holdout.csv", index=False)
pd.concat([train_df.assign(split="train"), holdout_df.assign(split="holdout")])[
    ["instance_id", "person_id", "offer_id", "time_received", "treated", "y_raw", "y", "split"] + FEATURES
].to_csv(OUTPUT_DIR / "phase3_model_table.csv", index=False)
cv_summary.to_csv(OUTPUT_DIR / "phase3_cv_results.csv")
print("Saved phase3_scored_holdout.csv", holdout_df[keep].shape, "| phase3_model_table.csv | phase3_cv_results.csv")

In [ ]:
# ============================================================
# PHASE 3 SUMMARY
# ============================================================
print("=" * 70); print("PHASE 3 SUMMARY"); print("=" * 70)
print(f"Instances modelled (BOGO + discount)  : {len(df):,}")
print(f"Train (first 20 days) / hold-out (last 10 days): {len(train_df):,} / {len(holdout_df):,}")
print(f"Treated share (viewed within 24h)     : {df['treated'].mean():.1%}")
print(f"Outcome                                : spend in hours [24, 72) after receipt, capped at ${CAP:.2f}")
print(f"Selected configs                       : {best_cfg}")
print()
display(results[["model", "Qini coef $/customer", "95% CI", "IPW Qini coef", "IPW 95% CI"]])
print("=" * 70)

# Phase 3 Conclusions

1. **Target.** Incremental revenue (spend in hours 24-72), capped at the training 99th percentile. Chosen because Phase 4 optimises dollars and because discounts change basket size as well as purchase frequency.
2. **Treatment.** Viewed the offer within 24 hours, evaluated with a landmark design so group membership never depends on the outcome. Accidental completers who never saw the offer remain control, as in Phase 1. The Phase 1 whole-window rule drifts from wave to wave for this reason.
3. **Algorithms.** A T-Learner (separate control and treated models) and an S-Learner (single model with treatment as a feature), both gradient-boosted trees.
4. **Validation.** Strictly time-based: train on the first 20 days, validate on the last 10. Hyper-parameters chosen by forward-chaining CV inside the training period. No random splits, and no outcome window crosses the train/validation boundary.
5. **Results.** See the hold-out table: both learners rank customers better than random targeting, with a monotone rise in observed uplift across predicted-uplift deciles.
6. **Placebo test.** Models trained on shuffled treatment labels show no ranking ability on the true hold-out.
7. **Limitations.**
   - Viewing is self-selected, so uplift is a *conditional association* adjusted for observed features, not a randomised causal effect. IPW and the placebo test check this but cannot remove unobserved confounding.
   - Customers who view the offer after hour 24 sit in the control group, which understates effects.
   - Later waves have more purchase history than training waves; per-day rates were used to limit this shift.
   - Only a 48-hour outcome window is used, so long-run effects are not captured.

**Outputs for Phase 4:** `phase3_scored_holdout.csv` (scored hold-out with offer reward cost) and `phase3_model_table.csv`.